# LOFAR Tied-Array Imaging of Type III Solar Radio Bursts — Implementation
## LOFAR Tied-Array를 이용한 Type III 태양 전파 폭발 — 구현

**Paper**: Morosan et al., A&A 568, A67 (2014). DOI: 10.1051/0004-6361/201423936

---

## Goals / 목표

**English.** This notebook reproduces the key physics and methodology of Morosan et al. (2014):
1. Plasma frequency as a function of coronal electron density for three density models (Newkirk, Mann, Zucca-ish).
2. Type III frequency drift f(t) for electron beams at various speeds.
3. Tied-array beam synthesis demonstration — phased-array beam pattern $W(\theta, \phi)$.
4. Synthetic Type III dynamic spectrum.
5. Electron beam travel time from the corona to 1 AU along a Parker spiral.
6. Fundamental vs harmonic plasma emission — optical depth comparison.

**한국어.** 이 노트북은 Morosan et al. (2014)의 핵심 물리와 방법론을 재현한다:
1. 세 가지 밀도 모델 (Newkirk, Mann, Zucca-유사)에 대한 코로나 전자 밀도의 함수로서의 플라스마 진동수.
2. 다양한 속도의 전자 빔에 대한 Type III 주파수 드리프트 f(t).
3. Tied-array 빔 합성 시연 — 위상 배열 빔 패턴 $W(\theta, \phi)$.
4. 합성 Type III 동적 스펙트럼.
5. 코로나에서 1 AU까지 Parker 나선을 따른 전자 빔 이동 시간.
6. 기본파 대 고조파 플라스마 방출 — 광학 깊이 비교.

In [ ]:
"""Imports and global constants.

This cell sets up the numerical libraries and physical constants used
throughout the notebook. All quantities follow CGS where convenient,
with electron densities in cm^-3 and frequencies in Hz/MHz.
"""

import numpy as np
import matplotlib.pyplot as plt

R_SUN_CM: float = 6.957e10  # Solar radius in cm
C_LIGHT_CM: float = 2.998e10  # Speed of light in cm/s
AU_CM: float = 1.496e13  # 1 Astronomical Unit in cm
C_PLASMA: float = 8980.0  # Hz * cm^(3/2); plasma-frequency constant

plt.rcParams.update({
    'figure.figsize': (9, 5),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

## 1. Plasma frequency vs coronal density profile / 플라스마 진동수 대 코로나 밀도 프로파일

**English.** The local plasma frequency is $f_p = 8980 \sqrt{N_e}$ Hz with $N_e$ in cm$^{-3}$. We compute $f_p(r)$ for three radial coronal density models and compare with the LOFAR LBA (30–90 MHz) and HBA (110–240 MHz) bands.

**한국어.** 국소 플라스마 진동수는 $f_p = 8980 \sqrt{N_e}$ Hz (여기서 $N_e$는 cm$^{-3}$)이다. 세 가지 방사 코로나 밀도 모델에 대해 $f_p(r)$을 계산하고 LOFAR LBA (30–90 MHz) 및 HBA (110–240 MHz) 대역과 비교한다.

In [ ]:
def newkirk_density(r_rsun: np.ndarray, fold: float = 1.0) -> np.ndarray:
    """Newkirk (1961) radial coronal electron-density model.

    n_e(r) = fold * 4.2e4 * 10^(4.32 / r)  [cm^-3], r in R_sun.

    Args:
        r_rsun: Heliocentric radius in solar radii.
        fold: Multiplicative fold factor (1 for normal streamer, 4 for dense).

    Returns:
        Electron density in cm^-3 at each r.
    """
    return fold * 4.2e4 * 10 ** (4.32 / r_rsun)


def mann_density(r_rsun: np.ndarray) -> np.ndarray:
    """Mann et al. (1999) hydrostatic coronal electron-density model.

    A simplified isothermal hydrostatic profile with T ~ 1.4 MK.

    Args:
        r_rsun: Heliocentric radius in solar radii.

    Returns:
        Electron density in cm^-3 at each r.
    """
    n0 = 5.14e9
    A = 13.6
    return n0 * np.exp(A * (1.0 / r_rsun - 1.0))


def zucca_like_density(r_rsun: np.ndarray) -> np.ndarray:
    """Simplified Zucca-like 1-D radial approximation for illustration.

    This is not the true 2-D DEM+pB map of Zucca et al. (2014) but a
    representative radial slice intermediate between Newkirk and Mann.

    Args:
        r_rsun: Heliocentric radius in solar radii.

    Returns:
        Electron density in cm^-3 at each r.
    """
    return 3.0e8 * np.exp(-5.0 * (r_rsun - 1.0))


def plasma_frequency(n_e: np.ndarray) -> np.ndarray:
    """Compute plasma frequency f_p = C * sqrt(N_e).

    Args:
        n_e: Electron density in cm^-3.

    Returns:
        Plasma frequency in Hz.
    """
    return C_PLASMA * np.sqrt(n_e)


r = np.linspace(1.01, 5.0, 400)
fp_newkirk = plasma_frequency(newkirk_density(r)) / 1e6
fp_mann = plasma_frequency(mann_density(r)) / 1e6
fp_zucca = plasma_frequency(zucca_like_density(r)) / 1e6

fig, ax = plt.subplots()
ax.plot(r, fp_newkirk, label='Newkirk (1961)')
ax.plot(r, fp_mann, label='Mann et al. (1999)')
ax.plot(r, fp_zucca, label='Zucca-like 1-D')
ax.axhspan(30, 90, alpha=0.15, color='C3', label='LOFAR LBA 30-90 MHz')
ax.axhspan(110, 240, alpha=0.15, color='C4', label='LOFAR HBA 110-240 MHz')
ax.set_xlabel('Heliocentric radius r [R_sun]')
ax.set_ylabel('Fundamental plasma frequency f_p [MHz]')
ax.set_yscale('log')
ax.set_ylim(1, 1000)
ax.set_title('Plasma frequency vs coronal height / 코로나 고도에 따른 플라스마 진동수')
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.show()

**English.** At 30 MHz (fundamental), Newkirk places the source near 2.2 R_sun, Mann at ~1.6 R_sun. Burst 2 in the paper is at ~4 R_sun at 30 MHz — far outside any of these radial models.

**한국어.** 30 MHz (기본파)에서, Newkirk는 소스를 ~2.2 R_sun 근처에, Mann은 ~1.6 R_sun에 놓는다. 논문의 Burst 2는 30 MHz에서 ~4 R_sun — 이러한 방사 모델의 예측을 크게 벗어난다.

## 2. Type III frequency drift for electron beams / 전자 빔의 Type III 주파수 드리프트

**English.** For a beam moving radially at constant speed $v_b$ through $n_e(r)$, we have $f(t) = n \cdot f_p(r(t))$ with $r(t) = r_0 + v_b t / R_\odot$. We simulate for $v_b = 0.1 c, 0.3 c, 0.5 c$ and plot the resulting drift tracks.

**한국어.** 일정 속도 $v_b$로 $n_e(r)$을 통과하는 빔의 경우 $f(t) = n \cdot f_p(r(t))$이며 $r(t) = r_0 + v_b t / R_\odot$이다. $v_b = 0.1 c, 0.3 c, 0.5 c$에 대해 시뮬레이션하고 결과 드리프트 경로를 플롯한다.

In [ ]:
def type3_drift(v_beam_c: float, r0_rsun: float = 1.5,
                t_max_s: float = 10.0, n_harmonic: int = 2,
                density_fn=newkirk_density) -> tuple:
    """Compute Type III drift curve f(t) for a radially moving electron beam.

    Args:
        v_beam_c: Beam speed in units of c.
        r0_rsun: Starting heliocentric radius in R_sun.
        t_max_s: Duration of the trajectory in seconds.
        n_harmonic: Plasma emission harmonic number (1=fundamental, 2=harmonic).
        density_fn: A callable n_e(r) in cm^-3.

    Returns:
        Tuple of (t_array_s, f_array_MHz, r_array_rsun).
    """
    t = np.linspace(0.0, t_max_s, 500)
    dr_per_s_rsun = v_beam_c * C_LIGHT_CM / R_SUN_CM
    r_t = r0_rsun + dr_per_s_rsun * t
    ne = density_fn(r_t)
    f_mhz = n_harmonic * plasma_frequency(ne) / 1e6
    return t, f_mhz, r_t


fig, ax = plt.subplots()
for v_c, colour in zip([0.1, 0.3, 0.5], ['C0', 'C1', 'C2']):
    t, f_mhz, r_t = type3_drift(v_c, r0_rsun=1.3, t_max_s=8.0)
    ax.plot(t, f_mhz, color=colour, label=f'v_b = {v_c:.1f}c')

ax.set_xlabel('Time since injection t [s]')
ax.set_ylabel('Emission frequency 2 f_p [MHz]')
ax.set_ylim(5, 200)
ax.set_yscale('log')
ax.axhspan(30, 90, alpha=0.1, color='C3', label='LOFAR LBA')
ax.set_title('Type III drift curves for different beam speeds (Newkirk profile, n=2)')
ax.legend()
plt.tight_layout()
plt.show()

# Compute drift rates numerically around f = 50 MHz
for v_c in [0.1, 0.3, 0.5]:
    t, f_mhz, _ = type3_drift(v_c, r0_rsun=1.3, t_max_s=8.0)
    idx = np.argmin(np.abs(f_mhz - 50.0))
    df_dt = np.gradient(f_mhz, t)[idx]
    print(f'  v_b = {v_c:.1f}c  ->  df/dt near 50 MHz = {df_dt:+.2f} MHz/s')

**English.** Faster beams produce steeper drift rates. At 0.3 c the drift rate near 50 MHz is about -5 to -10 MHz/s — consistent with the Morosan et al. distribution of -2 to -17 MHz/s at 30-60 MHz.

**한국어.** 빠른 빔이 가파른 드리프트 율을 생성한다. 0.3 c에서 50 MHz 부근의 드리프트 율은 약 -5 ~ -10 MHz/s — Morosan et al.의 30-60 MHz에서 -2 ~ -17 MHz/s 분포와 일치한다.

## 3. Tied-array beam synthesis demonstration / Tied-array 빔 합성 시연

**English.** We synthesize a tied-array beam pattern $W(\theta,\phi) = \sum_n w_n e^{i \mathbf{k}\cdot \mathbf{r}_n}$ for a 24-station array scaled to the LOFAR core (max baseline ~2 km) at 30 MHz. Complex weights steer the beam to pointing (θ₀, φ₀); here we plot magnitude squared vs sky offset for a zenith-steered beam and verify the expected FWHM of ~21' at 30 MHz.

**한국어.** 30 MHz에서 LOFAR 코어 규모(최대 기선 ~2 km)로 스케일된 24 스테이션 배열에 대해 tied-array 빔 패턴 $W(\theta,\phi) = \sum_n w_n e^{i \mathbf{k}\cdot \mathbf{r}_n}$을 합성한다. 복소 가중치가 빔을 (θ₀, φ₀)로 조향한다; 여기서는 천정 조향 빔에 대해 하늘 오프셋 대 크기 제곱을 플롯하고 30 MHz에서 예상되는 FWHM ~21'을 검증한다.

In [ ]:
def tied_array_beam(positions_m: np.ndarray, wavelength_m: float,
                    theta_arcmin: np.ndarray, phi_arcmin: np.ndarray,
                    pointing: tuple = (0.0, 0.0)) -> np.ndarray:
    """Synthesize a tied-array beam power pattern.

    Implements W(theta, phi) = sum_n w_n * exp(i k . r_n) with uniform weights.

    Args:
        positions_m: (N, 2) array of station x, y positions in metres.
        wavelength_m: Observing wavelength in metres.
        theta_arcmin: 1D array of offsets in one sky direction (arcmin).
        phi_arcmin: 1D array of offsets in the orthogonal sky direction (arcmin).
        pointing: (theta0, phi0) beam pointing in arcmin.

    Returns:
        2D array |W|^2 over the (theta, phi) sky grid, normalised to its peak.
    """
    arcmin_to_rad = np.pi / (180.0 * 60.0)
    theta_rad = theta_arcmin * arcmin_to_rad
    phi_rad = phi_arcmin * arcmin_to_rad
    theta0 = pointing[0] * arcmin_to_rad
    phi0 = pointing[1] * arcmin_to_rad
    k = 2.0 * np.pi / wavelength_m
    tg, pg = np.meshgrid(theta_rad, phi_rad, indexing='ij')
    W = np.zeros_like(tg, dtype=np.complex128)
    for x, y in positions_m:
        phase_target = k * (x * tg + y * pg)
        phase_point = k * (x * theta0 + y * phi0)
        W += np.exp(1j * (phase_target - phase_point))
    W /= len(positions_m)
    return np.abs(W) ** 2


rng = np.random.default_rng(seed=42)
N_stations = 24
baseline_m = 2000.0
positions = rng.uniform(-baseline_m / 2, baseline_m / 2, size=(N_stations, 2))

theta_grid = np.linspace(-40, 40, 200)
phi_grid = np.linspace(-40, 40, 200)

for freq_mhz in [30.0, 60.0, 90.0]:
    wl_m = 3e8 / (freq_mhz * 1e6)
    W_pow = tied_array_beam(positions, wl_m, theta_grid, phi_grid)
    # Measure FWHM along theta at phi = 0
    cut = W_pow[:, len(phi_grid) // 2]
    half_max = cut.max() / 2
    crossings = np.where(np.diff(np.sign(cut - half_max)))[0]
    if len(crossings) >= 2:
        fwhm = theta_grid[crossings[-1]] - theta_grid[crossings[0]]
    else:
        fwhm = float('nan')
    # Compare with theoretical 1.22 lambda / B
    fwhm_theory_arcmin = 1.22 * wl_m / baseline_m * (180 * 60 / np.pi)
    print(f'  f = {freq_mhz:.0f} MHz:  measured FWHM = {fwhm:.1f}\';  '
          f'1.22 lambda/B = {fwhm_theory_arcmin:.1f}\' ')

# Plot 30 MHz pattern in 2D
wl_30 = 3e8 / 30e6
W_30 = tied_array_beam(positions, wl_30, theta_grid, phi_grid)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(10 * np.log10(W_30 + 1e-6), extent=[-40, 40, -40, 40],
               origin='lower', cmap='viridis', vmin=-30, vmax=0)
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Beam power [dB]')
ax.set_xlabel('theta [arcmin]')
ax.set_ylabel('phi [arcmin]')
ax.set_title('Tied-array beam pattern at 30 MHz (24-station LOFAR core)')
plt.tight_layout()
plt.show()

**English.** The measured FWHM closely matches the classical $1.22 \lambda / B$ diffraction limit, validating the beam-synthesis calculation. Random station positions suppress grating lobes.

**한국어.** 측정된 FWHM은 고전적 회절 한계 $1.22 \lambda / B$와 밀접하게 일치하여 빔 합성 계산을 검증한다. 무작위 스테이션 위치가 격자 로브를 억제한다.

## 4. Dynamic spectrum — synthetic Type III burst / 동적 스펙트럼 — 합성 Type III 폭발

**English.** We generate a synthetic 30-minute LOFAR-like dynamic spectrum at 30-90 MHz with three Type III bursts of varying drift rates plus a background noise floor, mimicking Morosan et al. Fig. 1b.

**한국어.** 30-90 MHz에서 다양한 드리프트 율의 세 Type III 폭발과 배경 잡음 플로어를 갖는 합성 30분 LOFAR 유사 동적 스펙트럼을 생성하여 Morosan et al. Fig. 1b를 모사한다.

In [ ]:
def synthetic_dynamic_spectrum(t_s: np.ndarray, f_mhz: np.ndarray,
                                bursts: list, noise_level: float = 0.1) -> np.ndarray:
    """Build a 2D synthetic dynamic spectrum containing Type III bursts.

    Each burst is characterised by (t0, v_beam_c, r0, amplitude, width_s).

    Args:
        t_s: 1D array of time samples in seconds.
        f_mhz: 1D array of frequency samples in MHz.
        bursts: List of tuples (t0_s, v_beam_c, r0_rsun, amp, width_s).
        noise_level: Standard deviation of additive Gaussian noise.

    Returns:
        2D array of intensity (f, t) in arbitrary units.
    """
    rng_local = np.random.default_rng(seed=7)
    dyn_spec = rng_local.normal(0, noise_level, size=(len(f_mhz), len(t_s)))
    for t0_s, v_c, r0, amp, wid in bursts:
        t_rel = t_s - t0_s
        dr_per_s = v_c * C_LIGHT_CM / R_SUN_CM
        r_track = r0 + dr_per_s * np.maximum(t_rel, 0)
        f_track_mhz = 2.0 * plasma_frequency(newkirk_density(r_track)) / 1e6
        for i, f_i in enumerate(f_mhz):
            # Gaussian ridge around f_track with temporal envelope
            dist_f = np.abs(f_i - f_track_mhz)
            env_t = np.exp(-((t_rel) / wid) ** 2) * (t_rel >= -wid)
            ridge = amp * env_t * np.exp(-(dist_f / 3.0) ** 2)
            dyn_spec[i] += ridge
    return dyn_spec


t_s = np.linspace(0, 1800, 600)  # 30 min at 3 s cadence (illustrative)
f_mhz = np.linspace(30, 90, 120)
bursts_list = [
    (120.0, 0.5, 1.3, 1.0, 3.0),
    (300.0, 0.3, 1.2, 0.8, 4.0),
    (540.0, 0.55, 1.35, 1.2, 2.5),
    (900.0, 0.4, 1.25, 0.9, 3.5),
    (1200.0, 0.35, 1.3, 0.7, 4.0),
    (1500.0, 0.5, 1.2, 1.1, 3.0),
]
spec = synthetic_dynamic_spectrum(t_s, f_mhz, bursts_list, noise_level=0.15)

fig, ax = plt.subplots(figsize=(11, 4.5))
im = ax.imshow(spec, aspect='auto', origin='upper',
               extent=[t_s[0], t_s[-1], f_mhz[-1], f_mhz[0]],
               cmap='inferno', vmin=0, vmax=1.2)
plt.colorbar(im, ax=ax, label='Intensity [arb. units]')
ax.set_xlabel('Time [s]')
ax.set_ylabel('Frequency [MHz]')
ax.set_title('Synthetic LOFAR-like dynamic spectrum with Type III storm / 합성 Type III 스톰 동적 스펙트럼')
plt.tight_layout()
plt.show()

**English.** The synthetic spectrum reproduces the qualitative appearance of a Type III storm: a succession of high-to-low frequency drifts visible as diagonal streaks, similar to Fig. 1b of the paper.

**한국어.** 합성 스펙트럼은 Type III 스톰의 정성적 외관을 재현한다: 논문의 Fig. 1b와 유사하게 대각선 줄무늬로 나타나는 고-저 주파수 드리프트의 연속.

## 5. Electron beam travel from corona to 1 AU / 코로나에서 1 AU까지의 전자 빔 이동

**English.** A Type III electron beam travels from the low corona along open field lines through the heliosphere. Assuming a Parker-spiral path of length $L \approx 1.1$ AU at solar wind speed $v_{sw} = 450$ km/s, and a beam speed $v_b$, the travel time is $t = L / v_b$. We compute t for a range of $v_b$.

**한국어.** Type III 전자 빔은 저고도 코로나에서 개방 자기력선을 따라 헬리오스피어를 통과한다. 태양풍 속도 $v_{sw} = 450$ km/s에서 $L \approx 1.1$ AU의 Parker 나선 경로와 빔 속도 $v_b$를 가정하면 이동 시간은 $t = L / v_b$이다. $v_b$ 범위에 대해 t를 계산한다.

In [ ]:
def parker_spiral_length_au(v_sw_kms: float = 450.0, r_au: float = 1.0) -> float:
    """Approximate length of a Parker-spiral field line from the Sun to r_au.

    At 1 AU the spiral angle is ~45 degrees, so the path length is
    approximately r_au / cos(spiral_angle).

    Args:
        v_sw_kms: Solar wind speed in km/s.
        r_au: Heliocentric distance in AU.

    Returns:
        Field-line path length in AU.
    """
    omega_sun = 2.66e-6  # rad/s, solar equatorial rotation
    v_sw_cm = v_sw_kms * 1e5
    r_cm = r_au * AU_CM
    spiral_angle = np.arctan(omega_sun * r_cm / v_sw_cm)
    return r_au / np.cos(spiral_angle)


L_au = parker_spiral_length_au(v_sw_kms=450.0, r_au=1.0)
L_cm = L_au * AU_CM
v_beam_fracs = np.array([0.05, 0.1, 0.2, 0.3, 0.5])
t_travel_s = L_cm / (v_beam_fracs * C_LIGHT_CM)

print(f'Parker-spiral length at 1 AU: {L_au:.2f} AU')
print('Beam speed [c]     Travel time [min]')
for v_c, t_s_val in zip(v_beam_fracs, t_travel_s):
    print(f'      {v_c:.2f}              {t_s_val / 60:.1f}')

fig, ax = plt.subplots()
ax.plot(v_beam_fracs, t_travel_s / 60, 'o-')
ax.set_xlabel('Electron-beam speed v_b [c]')
ax.set_ylabel('Travel time to 1 AU [min]')
ax.set_title('Type III beam propagation: corona to Earth / 코로나에서 지구까지')
plt.tight_layout()
plt.show()

**English.** For a 0.3 c beam, the travel time is ~30 minutes; for 0.1 c, ~90 minutes. These values are observed in WIND/STEREO in-situ radio and Langmuir-wave detections following LOFAR coronal Type IIIs.

**한국어.** 0.3 c 빔의 경우 이동 시간은 ~30분; 0.1 c의 경우 ~90분이다. 이러한 값은 LOFAR 코로나 Type III 이후 WIND/STEREO 현장(in-situ) 전파 및 Langmuir 파 탐지에서 관측된다.

## 6. Fundamental vs harmonic plasma emission / 기본파 대 고조파 플라스마 방출

**English.** Free-free absorption optical depth is $\tau_\mathrm{ff} \propto n_e^2 / (T^{3/2} f^2) \cdot L$. Fundamental ($f = f_p$) suffers ~16× more absorption than harmonic ($f = 2 f_p$) at the same emission site. We compute $\tau$ for both.

**한국어.** 자유-자유 흡수 광학 깊이는 $\tau_\mathrm{ff} \propto n_e^2 / (T^{3/2} f^2) \cdot L$이다. 기본파 ($f = f_p$)는 동일 방출 지점에서 고조파 ($f = 2 f_p$)보다 ~16배 더 많은 흡수를 받는다. 둘 다에 대해 $\tau$를 계산한다.

In [ ]:
def free_free_tau(n_e: np.ndarray, T_K: float, f_hz: np.ndarray,
                  path_cm: float) -> np.ndarray:
    """Compute free-free (bremsstrahlung) optical depth.

    Standard CGS expression (approximate, Gaunt ~ 1):
        tau_ff = 0.2 * n_e^2 * T^(-3/2) * f^(-2) * L
    with n_e in cm^-3, T in K, f in Hz, L in cm.

    Args:
        n_e: Electron density in cm^-3.
        T_K: Electron temperature in K.
        f_hz: Frequency in Hz.
        path_cm: Path length through emitting plasma in cm.

    Returns:
        Dimensionless optical depth.
    """
    return 0.2 * n_e ** 2 * T_K ** (-1.5) * f_hz ** (-2) * path_cm


r_grid = np.linspace(1.1, 5.0, 200)
ne = newkirk_density(r_grid)
fp_hz = plasma_frequency(ne)
f_fund = fp_hz
f_harm = 2.0 * fp_hz
path_cm = 0.5 * R_SUN_CM  # typical emission region path length
T_corona = 1.4e6

tau_fund = free_free_tau(ne, T_corona, f_fund, path_cm)
tau_harm = free_free_tau(ne, T_corona, f_harm, path_cm)

fig, ax = plt.subplots()
ax.plot(r_grid, tau_fund, label=r'Fundamental (f = f_p)')
ax.plot(r_grid, tau_harm, label=r'Harmonic (f = 2 f_p)')
ax.axhline(1, color='k', linestyle='--', alpha=0.5, label=r'$\tau = 1$')
ax.set_xlabel('Heliocentric radius r [R_sun]')
ax.set_ylabel('Free-free optical depth tau_ff')
ax.set_yscale('log')
ax.set_title('Fundamental vs harmonic free-free absorption / 기본파 대 고조파 자유-자유 흡수')
ax.legend()
plt.tight_layout()
plt.show()

# Show the 4x-in-frequency factor -> 16x in tau
idx = np.argmin(np.abs(r_grid - 2.0))
print(f'At r = 2.0 R_sun: tau_fund = {tau_fund[idx]:.3e},  tau_harm = {tau_harm[idx]:.3e}')
print(f'Ratio tau_fund / tau_harm = {tau_fund[idx] / tau_harm[idx]:.2f}  (expected 4)')

**English.** Because $\tau \propto f^{-2}$ and harmonic emission has twice the frequency, $\tau_\mathrm{fund}/\tau_\mathrm{harm} = 4$ at the same emission site. Together with the longer group-velocity path near the plasma cutoff, this gives harmonic emission a significant advantage at high altitudes — justifying the $n=2$ assumption used by the paper.

**한국어.** $\tau \propto f^{-2}$이고 고조파 방출이 두 배 주파수를 가지므로 동일 방출 지점에서 $\tau_\mathrm{fund}/\tau_\mathrm{harm} = 4$이다. 플라스마 차단 근처의 긴 군속도 경로와 결합되어, 이는 고조파 방출에 높은 고도에서 상당한 이점을 제공한다 — 본 논문이 사용한 $n=2$ 가정을 정당화한다.

## 7. Summary / 요약

**English.** This notebook reproduced six key physics ingredients of Morosan et al. (2014):

1. Plasma-frequency-to-height mapping under three coronal density models.
2. Type III drift-rate curves for beams at 0.1-0.5 c.
3. Tied-array beam synthesis yielding FWHMs consistent with the LOFAR observation.
4. Synthetic dynamic spectrum illustrating a Type III storm.
5. Parker-spiral travel times: ~30 min for a 0.3 c beam to reach 1 AU.
6. Free-free optical depth comparison motivating the n=2 harmonic assumption at high altitudes.

The central unsolved puzzle — Burst 2 at 4 R_sun at 30 MHz requiring $n_e \sim 3\times 10^6$ cm$^{-3}$ unreachable by any 1-D radial model — remains the key motivation for CME-driven streamer compression scenarios in the paper.

**한국어.** 이 노트북은 Morosan et al. (2014)의 여섯 가지 핵심 물리 요소를 재현했다:

1. 세 가지 코로나 밀도 모델 하에서의 플라스마 진동수-고도 매핑.
2. 0.1-0.5 c 빔에 대한 Type III 드리프트 율 곡선.
3. LOFAR 관측과 일치하는 FWHM을 산출하는 tied-array 빔 합성.
4. Type III 스톰을 예시하는 합성 동적 스펙트럼.
5. Parker 나선 이동 시간: 0.3 c 빔이 1 AU에 도달하는 데 ~30분.
6. 고고도에서 n=2 고조파 가정을 정당화하는 자유-자유 광학 깊이 비교.

핵심 미해결 수수께끼 — 30 MHz에서 4 R_sun의 Burst 2가 어떤 1차원 방사 모델로도 도달할 수 없는 $n_e \sim 3\times 10^6$ cm$^{-3}$을 요구 — 는 본 논문의 CME 유도 스트리머 압축 시나리오의 핵심 동기로 남는다.